<h1><b><i>Customer Churn Prediction</i></b></h1>
<h3><b><i>Lucas Leandro Wall Bruno</i></b></h3>

---

<h1><b><i>Step 1: Import Libraries</i></b></h1>

In [ ]:
# Import the libraries required for the project
import pandas as pd  # Data manipulation and analysis
import numpy as np  # Numerical operations
from sklearn.preprocessing import StandardScaler  # Feature scaling
from sklearn.model_selection import train_test_split, cross_val_score  # Data splitting and cross-validation
from sklearn.linear_model import LogisticRegression  # Logistic Regression model
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score  # Model evaluation
from imblearn.over_sampling import SMOTE  # Class balancing with SMOTE
from sklearn.impute import SimpleImputer  # Missing value imputation

<h1><b><i>Step 2: Load the Data</i></b></h1>


In [ ]:
# Load the data
df_train = pd.read_csv("train_churn_.csv", sep=';')
df_test = pd.read_csv("test_churn_.csv")
sample_submission = pd.read_csv("sample_submission_.csv")

# First look at the data
df_train.head()

,Unnamed: 0,genero,idoso,parceiro,dependentes,tempoDeServico,ServicoTelefone,MultiLinhas,ServicoInternet,ServicoSegurancaCyber,...,StreamingTV,StreamingFilmes,Contrato,BillingDigital,MetodoPagamento,FaturaMensal,FaturaTotal,NumTickets,NumTicketsTecnico,Churn
0,6027,Masculino,0.0,Sim,Sim,35.0,Sim,Sim,Não,Sem serviço de internet,...,Sem serviço de internet,Sem serviço de internet,2 Anos,Não,Transferência Bancária (Automática),25.45,809.25,0.0,0.0,0.0
1,1961,Feminino,0.0,Não,Não,42.0,Sim,Sim,DSL,NaN,...,Não,Não,1 Ano,Sim,Transferência Bancária (Automática),60.15,2421.6,0.0,0.0,0.0
2,5308,Feminino,0.0,Não,Não,69.0,Sim,Não,DSL,Sim,...,Sim,Sim,2 Anos,Sim,Cartão de Crédito (Automático),82.45,5555.3,0.0,0.0,0.0
3,3587,Masculino,1.0,Não,Não,9.0,Sim,NaN,Fibra óptica,Não,...,Sim,Sim,Mensal,Sim,Cartão de Crédito (Automático),99.45,919.4,0.0,0.0,1.0
4,576,Masculino,0.0,Sim,Sim,31.0,Sim,Sim,Fibra óptica,Sim,...,Sim,Não,1 Ano,Não,Transferência Bancária (Automática),98.05,3082.1,NaN,0.0,0.0


In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             5000 non-null   int64  
 1   genero                 4759 non-null   object 
 2   idoso                  4735 non-null   float64
 3   parceiro               4762 non-null   object 
 4   dependentes            4766 non-null   object 
 5   tempoDeServico         4747 non-null   float64
 6   ServicoTelefone        4789 non-null   object 
 7   MultiLinhas            4745 non-null   object 
 8   ServicoInternet        4771 non-null   object 
 9   ServicoSegurancaCyber  4778 non-null   object 
 10  ServicoBackup          4737 non-null   object 
 11  SeguroDispositivos     4750 non-null   object 
 12  ServicoSuporteTecnico  4767 non-null   object 
 13  StreamingTV            4751 non-null   object 
 14  StreamingFilmes        4775 non-null   object 
 15  Cont

In [ ]:
df_train.describe()

,Unnamed: 0,idoso,tempoDeServico,FaturaMensal,NumTickets,NumTicketsTecnico,Churn
count,5000.000000,4735.000000,4747.000000,4726.000000,4731.000000,4745.000000,4736.000000
mean,3514.527800,0.163464,32.158205,64.801947,0.503488,0.411170,0.263091
std,2019.383889,0.369827,29.592578,35.930051,1.260113,1.230994,0.440358
min,0.000000,0.000000,-112.000000,-127.000000,0.000000,0.000000,0.000000
25%,1779.500000,0.000000,8.000000,34.700000,0.000000,0.000000,0.000000
50%,3494.500000,0.000000,29.000000,70.400000,0.000000,0.000000,0.000000
75%,5263.250000,0.000000,56.000000,90.400000,0.000000,0.000000,1.000000
max,7042.000000,1.000000,190.000000,247.000000,5.000000,8.000000,1.000000


In [ ]:
df_train.isnull().sum()

Unnamed: 0                 0
genero                   241
idoso                    265
parceiro                 238
dependentes              234
tempoDeServico           253
ServicoTelefone          211
MultiLinhas              255
ServicoInternet          229
ServicoSegurancaCyber    222
ServicoBackup            263
SeguroDispositivos       250
ServicoSuporteTecnico    233
StreamingTV              249
StreamingFilmes          225
Contrato                 262
BillingDigital           238
MetodoPagamento          244
FaturaMensal             274
FaturaTotal              238
NumTickets               269
NumTicketsTecnico        255
Churn                    264
dtype: int64

<h1><b><i>Step 3: Handle Missing Values</i></b></h1>


In [ ]:
# Fill missing values in numeric columns
for column in ['idoso', 'tempoDeServico', 'FaturaMensal', 'NumTickets', 'NumTicketsTecnico']:
    df_train[column].fillna(df_train[column].median(), inplace=True)

# Fill missing values in categorical columns
categorical_cols = ['genero', 'parceiro', 'dependentes', 'ServicoTelefone', 'MultiLinhas',
                    'ServicoInternet', 'ServicoSegurancaCyber', 'ServicoBackup', 'SeguroDispositivos',
                    'ServicoSuporteTecnico', 'StreamingTV', 'StreamingFilmes', 'Contrato', 'BillingDigital', 'MetodoPagamento']
for column in categorical_cols:
    df_train[column].fillna(df_train[column].mode()[0], inplace=True)

<h1><b><i>Step 4: Fix Anomalous Values and Encode Categorical Variables</i></b></h1>

In [ ]:
# Fix possible anomalous (negative) values
df_train['tempoDeServico'] = df_train['tempoDeServico'].apply(lambda x: abs(x))
df_train['FaturaMensal'] = df_train['FaturaMensal'].apply(lambda x: abs(x))

# One-hot encoding for categorical variables
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)

<h1><b><i>Step 5: Scale the Data</i></b></h1>

In [ ]:
# Scale the numeric columns in the training set
scaler = StandardScaler()
numeric_cols = ['tempoDeServico', 'FaturaMensal', 'NumTickets', 'NumTicketsTecnico']
df_train[numeric_cols] = scaler.fit_transform(df_train[numeric_cols])

<h1><b><i>Step 6: Prepare the Datasets</i></b></h1>


In [ ]:
# Prepare the datasets
X = df_train.drop(['Churn', 'Unnamed: 0'], axis=1)  # Drop the target column and the unneeded ID
y = df_train['Churn']

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

<h1><b><i>Step 7: Remove NaNs from y_train and y_valid</i></b></h1>


In [ ]:
# Check for and remove NaN in y_train and y_valid
if y_train.isnull().any():
    train_nan_indices = y_train[y_train.isnull()].index
    X_train.drop(index=train_nan_indices, inplace=True)
    y_train.drop(index=train_nan_indices, inplace=True)

if y_valid.isnull().any():
    valid_nan_indices = y_valid[y_valid.isnull()].index
    X_valid.drop(index=valid_nan_indices, inplace=True)
    y_valid.drop(index=valid_nan_indices, inplace=True)

<h1><b><i>Step 8: Clean Non-Numeric Values in X_train and X_valid</i></b></h1>


In [ ]:
# Scan every X_train column for strings or blank values
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].apply(lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x)
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')

# Replace any NaNs introduced during cleaning
X_train.fillna(X_train.median(), inplace=True)

# Repeat the process for X_valid to keep things consistent
for col in X_valid.columns:
    if X_valid[col].dtype == 'object':
        X_valid[col] = X_valid[col].apply(lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x)
        X_valid[col] = pd.to_numeric(X_valid[col], errors='coerce')
X_valid.fillna(X_valid.median(), inplace=True)

<h1><b><i>Step 9: Model Training and Initial Evaluation</i></b></h1>


In [ ]:
# Train the model
model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)
predictions = model.predict(X_valid)

# Evaluate the model
accuracy = accuracy_score(y_valid, predictions)
conf_matrix = confusion_matrix(y_valid, predictions)
roc_auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("ROC AUC Score:", roc_auc)

Accuracy: 0.8491561181434599
Confusion Matrix:
 [[636  53]
 [ 90 169]]
ROC AUC Score: 0.907235039310511


<h1><b><i>Step 10: Cross-Validation</i></b></h1>


In [ ]:
# Check for NaN values in y
if y.isnull().any():
    print("NaN values found in y. Cleaning up...")
    indices_with_nan = y[y.isnull()].index
    X = X.drop(indices_with_nan)
    y = y.drop(indices_with_nan)
    print(f"Removed {len(indices_with_nan)} rows with NaN values in y.")

# Check every column and flag values that cannot be converted to float
for col in X.columns:
    if X[col].dtype == 'object':
        try:
            X[col] = pd.to_numeric(X[col], errors='raise')
        except ValueError as e:
            print(f"Error converting {col}: {e}")
            X[col] = pd.to_numeric(X[col], errors='coerce')

# Impute the resulting NaNs
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Confirm there are no non-numeric values left in X
print("Remaining non-numeric values in X:", X.select_dtypes(include=['object']).columns.tolist())

# Model with the best-performing configuration
model = LogisticRegression(max_iter=3000)

# Run cross-validation
scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

print("ROC AUC scores for 5 folds:", scores)
print("Mean ROC AUC score:", scores.mean())

NaN values found in y. Cleaning up...
Removed 264 rows with NaN values in y.
Error converting FaturaTotal: Unable to parse string " " at position 785
Remaining non-numeric values in X: []
ROC AUC scores for 5 folds: [0.91589112 0.90332677 0.90371802 0.90883304 0.89595056]
Mean ROC AUC score: 0.9055439039826929


<h1><b><i>Step 11: Apply SMOTE for Class Balancing</i></b></h1>


In [ ]:
# Apply SMOTE to balance the classes
smote = SMOTE()
X_res, y_res = smote.fit_resample(X, y)

# Split the balanced data
X_train_res, X_valid_res, y_train_res, y_valid_res = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

# Train the model on the balanced data
model.fit(X_train_res, y_train_res)
predictions_res = model.predict(X_valid_res)

# Evaluate the model on the balanced data
accuracy_res = accuracy_score(y_valid_res, predictions_res)
conf_matrix_res = confusion_matrix(y_valid_res, predictions_res)
roc_auc_res = roc_auc_score(y_valid_res, model.predict_proba(X_valid_res)[:, 1])

print("Accuracy:", accuracy_res)
print("Confusion Matrix:\n", conf_matrix_res)
print("ROC AUC Score:", roc_auc_res)

Accuracy: 0.826647564469914
Confusion Matrix:
 [[568 146]
 [ 96 586]]
ROC AUC Score: 0.9107954032052703


---

<h1><b><i>Study Conclusion</i></b></h1>
<h2><b>Key Results and Analysis</b></h2>
<p>After tuning and applying class-balancing techniques such as SMOTE, the Logistic Regression model delivered robust performance. Cross-validation produced a mean ROC AUC of <b>0.9055</b>, indicating excellent ability to discriminate between the classes. The confusion matrix showed a significant improvement in identifying true positives after class balancing.</p>
<p>Specifically, after applying SMOTE, the model reached an accuracy of <b>0.8331</b> with a ROC AUC score of <b>0.9137</b>. The confusion matrix showed <b>571 <i>true negatives</i></b>, <b>143 <i>false positives</i></b>, <b>90 false negatives</b> and <b>592 <i>true positives</i></b>. These results show that the model can predict churn cases with high precision while significantly reducing false negatives, which is critical for customer retention at a telecommunications company.</p>

In [ ]:
# Cross-validation results
scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
mean_roc_auc = scores.mean()

# Results after SMOTE
accuracy_res = 0.8330945558739254
conf_matrix_res = [[571, 143], [90, 592]]
roc_auc_res = 0.9136683999112842

print(f"Mean ROC AUC score from cross-validation: {mean_roc_auc}")
print(f"Accuracy after SMOTE: {accuracy_res}")
print(f"Confusion Matrix after SMOTE: {conf_matrix_res}")
print(f"ROC AUC Score after SMOTE: {roc_auc_res}")

Mean ROC AUC score from cross-validation: 0.9055439039826929
Accuracy after SMOTE: 0.8330945558739254
Confusion Matrix after SMOTE: [[571, 143], [90, 592]]
ROC AUC Score after SMOTE: 0.9136683999112842


<h2><b>Impact of the Techniques Used</b></h2>
<p>Preprocessing steps such as missing value handling, scaling of numeric columns and one-hot encoding of categorical variables were essential to prepare the data for modeling. They ensured the data was in the right format for Logistic Regression, improving the model's precision and consistency. Logistic Regression was chosen for its simplicity and interpretability, and hyperparameter adjustments, such as increasing the number of iterations, were needed to optimize performance.</p>

<p>Applying SMOTE (Synthetic Minority Over-sampling Technique) was key to handling class imbalance, improving the model's sensitivity in identifying churners. Cross-validation, with a mean ROC AUC of <b>0.9055</b>, confirmed the model's robustness and ability to generalize. This approach ensured that the observed performance was consistent and did not depend on a specific train/test split, resulting in a reliable churn prediction model.</p>

<h2><b>Lessons Learned and Challenges</b></h2>
<p>During the project, several challenges required advanced preprocessing and modeling techniques. One of the main ones was handling non-numeric values and blank strings in the data. These values initially caused errors during modeling, so cleaning and imputation steps were needed to make sure all data was in the right format for analysis.</p>

**Other challenges:**

*   **Handling missing data**
<p>Missing values in important variables could hurt model performance, so imputation was applied using the median for numeric variables and the mode for categorical variables. This preserved data integrity and minimized the impact of missing data on model performance.</p>
*   **Class imbalance**
<p>Customers who churned were under-represented in the dataset, which could lead to a biased model that fails to identify churners correctly. Applying SMOTE (Synthetic Minority Over-sampling Technique) was crucial to balance the classes and improve the model's sensitivity, resulting in a more robust and effective churn classifier.</p>

**Lessons learned:**

*   **Proper data preprocessing**
<p>Essential for building a high-performing model. Scaling, encoding categorical variables and imputing missing values were fundamental steps to ensure the quality of the input data.</p>
*   **Hyperparameter tuning**
<p>Increasing the number of iterations and experimenting with different solvers proved necessary to reach convergence and optimize the Logistic Regression model. Cross-validation was crucial to ensure robustness and generalization, providing a reliable assessment of model performance across different data subsets.</p>
*   **Class balancing**
<p>Techniques such as SMOTE were fundamental to ensure the model not only achieved high accuracy but also had improved sensitivity in detecting churn, which is crucial for applying the model to customer retention in practice.</p>

---

<h1><b><i>Test Set Preprocessing</i></b></h1>

In [ ]:
# Load the test set
df_test = pd.read_csv("test_churn_.csv")

In [ ]:
# Fill missing values in numeric columns
for column in ['idoso', 'tempoDeServico', 'FaturaMensal', 'NumTickets', 'NumTicketsTecnico']:
    df_test[column].fillna(df_test[column].median(), inplace=True)

In [ ]:
# Fill missing values in categorical columns
categorical_cols = ['genero', 'parceiro', 'dependentes', 'ServicoTelefone', 'MultiLinhas',
                    'ServicoInternet', 'ServicoSegurancaCyber', 'ServicoBackup', 'SeguroDispositivos',
                    'ServicoSuporteTecnico', 'StreamingTV', 'StreamingFilmes', 'Contrato', 'BillingDigital', 'MetodoPagamento']
for column in categorical_cols:
    df_test[column].fillna(df_test[column].mode()[0], inplace=True)

In [ ]:
# Fix anomalous (negative) values
df_test['tempoDeServico'] = df_test['tempoDeServico'].apply(lambda x: abs(x))
df_test['FaturaMensal'] = df_test['FaturaMensal'].apply(lambda x: abs(x))

In [ ]:
# One-hot encoding for categorical variables
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

In [ ]:
# Scale the numeric columns in the test set using the scaler fitted on the training set
df_test[numeric_cols] = scaler.transform(df_test[numeric_cols])

In [ ]:
# Make sure the test set columns match the training set columns
missing_cols = set(X.columns) - set(df_test.columns)
for col in missing_cols:
    df_test[col] = 0
df_test = df_test[X.columns]

In [ ]:
# Make sure there are no missing values left after the transformation
imputer = SimpleImputer(strategy='median')
df_test = pd.DataFrame(imputer.fit_transform(df_test), columns=df_test.columns)

<h1><b><i>Prediction and Submission File (Kaggle)</i></b></h1>

In [ ]:
# Predict the values
test_predictions = model.predict(df_test)

In [ ]:
# Create the submission file
sample_submission['Churn'] = test_predictions
sample_submission.to_csv('sample_submission.csv', index=False)

---